# Laboratorio 23. Clasificación de imágenes para inspección visual de calidad

**Pregunta guía:** ¿Puede una CNN detectar rayas y manchas en imágenes de productos mejor que un clasificador lineal, y cómo se convierte su salida en una decisión de inspección que minimice el costo?

## Objetivos
Al finalizar podrás generar y particionar un conjunto de imágenes desbalanceado; entrenar un clasificador lineal, una CNN y una CNN con aumento de datos; evaluar con matriz de confusión, *recall* por clase y F1 macro; elegir un umbral de rechazo según costos de negocio; y explicar predicciones con mapas de saliencia.

## Prerrequisitos y conexión
Usa convoluciones y CNN (laboratorios 12 y 13) y la evaluación con train, validación y test (laboratorio 19). Cada notebook puede ejecutarse de forma independiente desde un kernel limpio.

## Teoría breve
Una CNN aplica filtros locales con parámetros compartidos; con *pooling* global la decisión no depende de **dónde** está el defecto. El aumento de datos aplica transformaciones que no cambian la etiqueta (giros y reflexiones: una raya sigue siendo raya) y enseña esa invariancia explícitamente. Para decidir, usamos la probabilidad de defecto $s=1-P(\text{ok}\mid x)$ y rechazamos la pieza si $s\ge\tau$; el umbral $\tau$ se elige minimizando $C=c_{FN}\cdot FN+c_{FP}\cdot FP$.

## Problema y datos
Una planta ficticia fotografía cada pieza en una cinta. Generamos 3 000 imágenes de 24×24 píxeles con iluminación y textura variables: 80 % sin defecto, 10 % con una raya fina (ángulo y posición aleatorios) y 10 % con una mancha difusa. Todos los datos se generan en el notebook.

## Perspectiva de negocio y datos
Dejar pasar una pieza defectuosa (falso negativo) cuesta un reclamo o una devolución; rechazar una buena (falso positivo) cuesta una revisión manual. La exactitud global oculta esa asimetría: con 80 % de piezas buenas, un modelo que aprueba todo tiene 80 % de exactitud y no detecta nada. Además, las imágenes etiquetadas son un activo de datos: su calidad y cobertura de casos raros limitan al modelo.

## Experimento conceptual
¿Cuánto aportan la estructura convolucional y el aumento de datos frente a un modelo lineal sobre píxeles? Fija la semilla, ejecuta las celdas en orden y cambia una decisión cada vez.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional. Usamos `rng` (generador de NumPy) para los datos y `torch.manual_seed` antes de cada entrenamiento para que las comparaciones entre modelos sean justas.

In [ ]:
import copy
import time

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 2026
rng = np.random.default_rng(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)

**Resultado e interpretación.** La celda confirma el entorno utilizado. El laboratorio está diseñado para CPU y se ejecuta en pocos minutos; una GPU disponible se utiliza sin ser requisito.

### Generar imágenes de piezas
La iluminación cambia entre fotos (un gradiente con nivel aleatorio) y la superficie tiene textura. Los defectos tienen contraste moderado para que no sean triviales.

In [ ]:
n_images, size = 3000, 24
class_names = ['ok', 'raya', 'mancha']
labels = rng.choice(3, n_images, p=[0.80, 0.10, 0.10])
yy, xx = np.mgrid[0:size, 0:size].astype(np.float32)


def make_image(label):
    image = rng.uniform(0.35, 0.65) + rng.uniform(-0.008, 0.008) * (xx + yy)  # iluminación
    image = image + rng.normal(0, 0.06, (size, size))                          # textura
    if label == 1:  # raya: segmento fino con ángulo, largo y posición aleatorios
        angle, length = rng.uniform(0, np.pi), rng.uniform(8, 16)
        cx, cy = rng.uniform(6, 18, 2)
        t = np.linspace(-length / 2, length / 2, 60)
        px = np.clip(np.round(cx + t * np.cos(angle)).astype(int), 0, size - 1)
        py = np.clip(np.round(cy + t * np.sin(angle)).astype(int), 0, size - 1)
        image[py, px] -= rng.uniform(0.15, 0.30)
    elif label == 2:  # mancha: zona circular más oscura y difusa
        cx, cy = rng.uniform(5, 19, 2)
        radius = rng.uniform(1.5, 3.5)
        image = image - rng.uniform(0.15, 0.30) * np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * radius ** 2))
    return np.clip(image, 0, 1).astype(np.float32)


images = np.stack([make_image(label) for label in labels])
order = rng.permutation(n_images)
train_idx, val_idx, test_idx = order[:1800], order[1800:2400], order[2400:]
print('imágenes:', images.shape, '| conteo por clase:', dict(zip(class_names, np.bincount(labels).tolist())))

fig, axes = plt.subplots(3, 6, figsize=(9, 4.8))
for row, label in enumerate(range(3)):
    for col, index in enumerate(np.flatnonzero(labels == label)[:6]):
        axes[row, col].imshow(images[index], cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
    axes[row, 0].set_title(class_names[label], loc='left', fontsize=9)
plt.tight_layout()
plt.show()

**Resultado e interpretación.** Las diferencias de iluminación entre piezas buenas son del mismo orden que el contraste de algunos defectos: un umbral sobre el brillo medio no basta. Las clases están desbalanceadas, como en una línea real.

### Normalizar y preparar tensores
Normalizamos con media y desviación de los píxeles de train. Las imágenes tienen forma (n, canal, alto, ancho).

In [ ]:
pixel_mean, pixel_std = images[train_idx].mean(), images[train_idx].std()


def to_tensors(idx):
    x = torch.tensor((images[idx] - pixel_mean) / pixel_std, dtype=torch.float32, device=device).unsqueeze(1)
    return x, torch.tensor(labels[idx], dtype=torch.long, device=device)


splits = {name: to_tensors(idx) for name, idx in [('train', train_idx), ('val', val_idx), ('test', test_idx)]}
print('train:', tuple(splits['train'][0].shape), '| clases en train:', np.bincount(labels[train_idx]))
print('baseline (todo ok) exactitud en validación:', round(float((labels[val_idx] == 0).mean()), 3))

**Resultado e interpretación.** El baseline que aprueba todas las piezas tiene alta exactitud y *recall* cero para ambos defectos. Esa es la referencia que demuestra por qué no reportaremos solo exactitud.

### Modelos: lineal, CNN y aumento de datos
El modelo lineal asigna un peso a cada píxel: debería aprender *dónde* suelen estar los defectos, lo que no sirve si aparecen en cualquier lugar. La CNN usa *max pooling* global: responde si el patrón aparece en cualquier posición.

In [ ]:
class LinearClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(size * size, 3)

    def forward(self, x):
        return self.linear(x.flatten(1))


class DefectCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),   # 12x12
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),  # 6x6
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveMaxPool2d(1))              # global
        self.classifier = nn.Linear(32, 3)

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))


def augment(x):
    # Giros y reflexiones no cambian la clase. Se aplica la misma transformación a todo el batch.
    if torch.rand(1).item() < 0.5:
        x = torch.flip(x, dims=[3])
    if torch.rand(1).item() < 0.5:
        x = torch.flip(x, dims=[2])
    return torch.rot90(x, int(torch.randint(0, 4, (1,)).item()), dims=[2, 3])


for cls in (LinearClassifier, DefectCNN):
    print(cls.__name__, '| parámetros:', sum(p.numel() for p in cls().parameters()))

**Resultado e interpretación.** La CNN tiene menos parámetros que el modelo lineal y, aun así, más capacidad para este problema: comparte filtros en todas las posiciones. Ese es su sesgo inductivo.

### Entrenar los tres modelos con el mismo protocolo
Misma semilla, optimizador y *early stopping* según la pérdida de validación. La única diferencia del tercer modelo es `augment` en los batches de train.

In [ ]:
def train_classifier(build_model, use_augmentation=False, epochs=50, lr=3e-3, batch_size=64, patience=8):
    torch.manual_seed(SEED)
    model = build_model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(*splits['train']), batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))
    x_val, y_val = splits['val']
    curve, best_loss, best_state, wait = [], np.inf, None, 0
    for epoch in range(epochs):
        model.train()
        for x, target in loader:
            if use_augmentation:
                x = augment(x)
            loss = F.cross_entropy(model(x), target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = F.cross_entropy(model(x_val), y_val).item()
        curve.append(val_loss)
        if val_loss < best_loss - 1e-4:
            best_loss, best_state, wait = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model, curve


def predict_proba(model, split):
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(splits[split][0]), dim=1).cpu().numpy()


configs = {'lineal': (LinearClassifier, False), 'cnn': (DefectCNN, False), 'cnn_aumento': (DefectCNN, True)}
models, curves = {}, {}
for name, (cls, use_augmentation) in configs.items():
    start = time.perf_counter()
    models[name], curves[name] = train_classifier(cls, use_augmentation)
    print(f'{name:12s} {len(curves[name])} épocas | {time.perf_counter() - start:.1f} s')

for name, curve in curves.items():
    plt.plot(curve, label=name)
plt.xlabel('época')
plt.ylabel('entropía cruzada (validación)')
plt.legend()
plt.show()

**Resultado e interpretación.** El aumento de datos suele hacer la pérdida de validación más estable y retrasar el sobreajuste, a cambio de más épocas. Si el modelo lineal se estanca en una pérdida alta, es porque no puede representar "defecto en cualquier lugar".

### Matriz de confusión y métricas por clase
Calculamos *recall* (qué fracción de cada clase se detecta), precisión y F1 macro en validación, y elegimos el modelo por F1 macro.

In [ ]:
def confusion(y_true, y_pred, n_classes=3):
    matrix = np.zeros((n_classes, n_classes), dtype=int)
    np.add.at(matrix, (y_true, y_pred), 1)
    return matrix


def class_metrics(y_true, y_pred):
    matrix = confusion(y_true, y_pred)
    recall = matrix.diagonal() / matrix.sum(axis=1).clip(min=1)
    precision = matrix.diagonal() / matrix.sum(axis=0).clip(min=1)
    f1 = 2 * precision * recall / (precision + recall).clip(min=1e-9)
    return {'exactitud': round(float((y_true == y_pred).mean()), 4), 'f1_macro': round(float(f1.mean()), 4),
            **{f'recall_{name}': round(float(r), 3) for name, r in zip(class_names, recall)}}


validation = {name: class_metrics(labels[val_idx], predict_proba(model, 'val').argmax(1))
              for name, model in models.items()}
for name, metrics in validation.items():
    print(f'{name:12s} {metrics}')

chosen = max(validation, key=lambda name: validation[name]['f1_macro'])
test_pred = predict_proba(models[chosen], 'test').argmax(1)
print(f'\nElegido por validación: {chosen} | test: {class_metrics(labels[test_idx], test_pred)}')

matrix = confusion(labels[test_idx], test_pred)
plt.imshow(matrix, cmap='Blues')
for i in range(3):
    for j in range(3):
        plt.text(j, i, matrix[i, j], ha='center', va='center')
plt.xticks(range(3), class_names)
plt.yticks(range(3), class_names)
plt.xlabel('predicción')
plt.ylabel('real')
plt.title(f'Matriz de confusión en test ({chosen})')
plt.show()

**Resultado e interpretación.** Lee la matriz por filas: cada fila es una clase real. Los errores más caros están en la columna "ok" de las filas de defectos (piezas defectuosas aprobadas). Confundir raya con mancha es menos grave: en ambos casos la pieza se rechaza.

### De probabilidades a una decisión de inspección
El modelo entrega probabilidades; la planta necesita aprobar o rechazar. Suponemos que dejar pasar un defecto cuesta 50 y revisar una pieza buena cuesta 5 (unidades monetarias por pieza). Elegimos el umbral en validación y lo aplicamos en test.

In [ ]:
COST_FN, COST_FP = 50.0, 5.0


def inspection_cost(y_true, defect_score, threshold):
    flagged = defect_score >= threshold
    is_defect = y_true != 0
    false_negatives = int((is_defect & ~flagged).sum())
    false_positives = int((~is_defect & flagged).sum())
    return (COST_FN * false_negatives + COST_FP * false_positives) / len(y_true), false_negatives, false_positives


thresholds = np.linspace(0.01, 0.99, 99)
val_score = 1 - predict_proba(models[chosen], 'val')[:, 0]
val_costs = [inspection_cost(labels[val_idx], val_score, t)[0] for t in thresholds]
best_threshold = float(thresholds[int(np.argmin(val_costs))])

test_score = 1 - predict_proba(models[chosen], 'test')[:, 0]
for label, threshold in [('umbral fijo 0,5', 0.5), ('umbral por costo', best_threshold)]:
    cost, fn, fp = inspection_cost(labels[test_idx], test_score, threshold)
    print(f'{label:28s} umbral={threshold:.2f} | costo medio por pieza={cost:.3f} | FN={fn} | FP={fp}')

plt.plot(thresholds, val_costs)
plt.axvline(best_threshold, color='black', linewidth=0.8)
plt.xlabel('umbral de rechazo τ')
plt.ylabel('costo medio por pieza (validación)')
plt.title('Costo de inspección según el umbral')
plt.show()

**Resultado e interpretación.** Con costos asimétricos, el umbral óptimo suele ser menor que 0,5: conviene rechazar algunas piezas buenas para no dejar pasar defectos. El umbral es una **decisión de negocio** que se revisa si cambian los costos, aunque el modelo sea el mismo.

### ¿En qué píxeles se fija el modelo?
Un mapa de saliencia es el valor absoluto del gradiente de la salida respecto de cada píxel: indica qué píxeles cambiarían más la predicción. Si el modelo es confiable, debería concentrarse en el defecto y no en la iluminación.

In [ ]:
model = models[chosen]
model.eval()
defect_examples = [i for i in range(len(test_idx)) if labels[test_idx][i] != 0][:4]
fig, axes = plt.subplots(2, 4, figsize=(9, 4.6))
for col, i in enumerate(defect_examples):
    x = splits['test'][0][i:i + 1].clone().requires_grad_(True)
    logits = model(x)
    predicted = int(logits.argmax(1).item())
    logits[0, predicted].backward()
    saliency = x.grad.abs()[0, 0].cpu().numpy()
    axes[0, col].imshow(images[test_idx][i], cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f'real={class_names[labels[test_idx][i]]}\npred={class_names[predicted]}', fontsize=8)
    axes[1, col].imshow(saliency, cmap='hot')
    for axis in axes[:, col]:
        axis.axis('off')
plt.suptitle('Imagen (arriba) y saliencia (abajo)')
plt.tight_layout()
plt.show()

**Resultado e interpretación.** Si la saliencia se concentra sobre la raya o la mancha, el modelo usa la evidencia correcta. Si se dispersa por toda la imagen o en los bordes, puede depender de artefactos (iluminación, fondo), algo que en datos reales ocurre con frecuencia y que una métrica alta no revela.

## Limitaciones y conclusiones clave
Las imágenes son sintéticas, pequeñas y en escala de grises. En inspección real se usan imágenes de mayor resolución, *transfer learning* desde modelos preentrenados (laboratorio 14) y conjuntos etiquetados por expertos; los defectos nuevos no vistos en train requieren monitoreo y, a menudo, detección de anomalías (laboratorio 24).

- La estructura convolucional y el *pooling* global hacen al modelo invariante a la posición del defecto.
- El aumento de datos incorpora invariancias conocidas del dominio sin etiquetar más imágenes.
- La decisión final depende de un umbral elegido con costos de negocio, no solo del *argmax*.

## Ejercicios progresivos
1. **Guiado:** cambia `COST_FN` a 10 y a 200 y observa cómo se mueve el umbral óptimo y el número de piezas rechazadas en test.
2. **Independiente:** entrena la CNN con pesos de clase en `F.cross_entropy(..., weight=...)` inversamente proporcionales a la frecuencia en train y compara el *recall* de defectos y el costo de inspección.
3. **Desafío:** reduce el set de entrenamiento a 300 imágenes (sin tocar validación ni test) y compara CNN con y sin aumento de datos. Explica por qué el aumento ayuda más con pocos datos.

## Conexión con el siguiente laboratorio
¿Qué pasa con un defecto que nunca se etiquetó? En el laboratorio 24 detectamos casos anómalos sin etiquetas usando un *autoencoder*.